# Ingestion

In [ ]:
%pip install beautifulsoup4==4.11.1 openai==2.41.1 llama-index==0.14.23 llama-index-embeddings-openai==0.6.0

## 1. Website network -> (Url,HTML) dictionary


### Method 1: Recursively search for websites from an initial URL with a depth limit

In [25]:
import requests
import time
import random
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/125.0.0.0 Safari/537.36"
)

def _crawl(
    current_url: str,
    depth: int,
    max_depth: int,
    root_domain: str,
    html_pages: dict,
    user_agent=USER_AGENT,
    min_delay=0.5,
    max_delay=1.5,
):
    if current_url in html_pages:
        return

    print(f"Crawling website: {current_url}...")
    
    # Add randomness to simulate human behaviour
    time.sleep(random.uniform(min_delay, max_delay))

    try:
        response = requests.get(
            current_url,
            timeout=10,
            headers={"User-Agent": user_agent},
        )
        response.raise_for_status()

        html = response.text
        html_pages[current_url] = html

        if depth == max_depth:
            return

        soup = BeautifulSoup(html, "html.parser")
        website_links = soup.find_all("a", href=True)

        for link in website_links:
            absolute_url = urljoin(current_url, link["href"])
            parsed_url = urlparse(absolute_url)

            # Stay within the same domain
            if parsed_url.netloc == root_domain:
                cleaned_url = parsed_url.geturl()


                _crawl(
                    current_url=cleaned_url,
                    depth=depth + 1,
                    max_depth=max_depth,
                    root_domain=root_domain,
                    html_pages=html_pages,
                )

    except Exception as e:
        print(f"Error crawling {current_url}: {e}")


def crawl_html(
    url: str, 
    max_depth: int,
    html_pages: dict[str, str] | None = None,
):
    if html_pages is None:
        html_pages = {}
    
    root_domain = urlparse(url).netloc

    _crawl(
        current_url=url,
        depth=0,
        max_depth=max_depth,
        root_domain=root_domain,
        html_pages=html_pages,
    )

    return html_pages

In [5]:
INITIAL_CRAWLED_URL = "https://nau64.com"
DEPTH = 2

In [ ]:
pages = crawl_html(INITIAL_CRAWLED_URL, max_depth=DEPTH)

In [14]:
len(pages)

60

### Method 2: Use the property of the website being paginated

In [26]:
CRAWLED_DATA_PATH = "crawled_data.csv"

In [12]:
import csv


def load_html_pages(csv_file: str) -> dict[str, str]:
    """
    Loads an existing HTML CSV into the format expected by _crawl().
    """

    html_pages = {}

    with open(csv_file, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            html_pages[row["url"]] = row["raw_text"]

    return html_pages

In [19]:
try:
    html_pages = pages
except:
    html_pages = load_html_pages(CRAWLED_DATA_PATH)

In [31]:
def crawl_paginated_html(
    url_template: str,
    html_pages=html_pages,
    max_depth=2,
    start_page=1,
    max_iters=10,
):
    """
    Crawls paginated websites while reusing a previously crawled CSV.

    Returns:
        dict[str, str]: {url: html}
    """

    for page in range(start_page, max_iters + 1):
        current_url = url_template.format(i=page)
        del html_pages[current_url]

        print(f"Checking page {page}: {current_url}")

        crawl_html(
            url=current_url,
            max_depth=max_depth,
            html_pages=html_pages,
        )

    return html_pages

In [32]:
URL_TEMPLATE="https://nau64.com/page/{i}/"
new_html_pages = crawl_paginated_html(URL_TEMPLATE)

Checking page 1: https://nau64.com/page/1/
Crawling website: https://nau64.com/page/1/...
Checking page 2: https://nau64.com/page/2/
Crawling website: https://nau64.com/page/2/...
Checking page 3: https://nau64.com/page/3/
Crawling website: https://nau64.com/page/3/...
Checking page 4: https://nau64.com/page/4/
Crawling website: https://nau64.com/page/4/...
Crawling website: https://nau64.com/emanuel-jesus-se-llevo-el-xxi-aniversario-open-de-nau64/...
Crawling website: https://nau64.com/emanuel-jesus-se-llevo-el-xxi-aniversario-open-de-nau64/#content...
Crawling website: https://nau64.com/matteo-reyes-se-llevo-el-xxi-aniversario-sub-14-de-nau64/...
Crawling website: https://nau64.com/el-campeon-sub-20-se-llevo-el-xxi-aniversario-blitz-de-nau64/...
Crawling website: https://nau64.com/el-campeon-sub-20-se-llevo-el-xxi-aniversario-blitz-de-nau64/#content...
Crawling website: https://nau64.com/benjamin-martello-se-consagro-campeon-del-sabatino-rapido-de-nau64/...
Crawling website: https://

## 2. (URL,HTML) dictionary -> CSV

In [33]:
import csv


def save_to_csv(
    html_by_url: dict[str, str],
    output_file: str,
):
    with open(output_file, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)

        writer.writerow(["url", "raw_text"])
        for url, html in html_by_url.items():
            writer.writerow([url, html])

In [34]:
save_to_csv(html_pages, CRAWLED_DATA_PATH)

## 3.1. CSV (URL, HTML) -> Cleaned CSV

In [35]:
import csv
import re
from urllib.parse import urlsplit, urlunsplit

from bs4 import BeautifulSoup


def extract_tag(html: str, tag_name="main") -> str:
    match = re.search(
        fr"<{tag_name}[^>]*>(.*?)</{tag_name}>",
        html,
        flags=re.DOTALL,
    )

    if match:
        return match.group(1)

    return ""


def extract_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    return soup.get_text()


def format_events(text: str) -> str:
    """
    Ensures exactly two newlines before every "[Event" marker.
    """
    return re.sub(r"\n*\s*(?=\[Event )", "\n\n", text)


def normalize_url(url: str) -> str:
    """
    Removes the fragment (#...) from a URL.

    Example:
    https://example.com/page#dashboard
    ->
    https://example.com/page
    """
    parts = urlsplit(url)
    return urlunsplit((parts.scheme, parts.netloc, parts.path, parts.query, ""))


def remove_odd_characters(text: str, odd_characters=("«", "»")):
    for odd_character in odd_characters:
        text = text.replace(odd_character, "")

    return text


def clean_html_csv(
    input_csv: str,
    output_csv: str,
):
    seen_urls = set()

    with open(input_csv, "r", encoding="utf-8", newline="") as infile, \
         open(output_csv, "w", encoding="utf-8", newline="") as outfile:

        reader = csv.DictReader(infile)
        fieldnames = list(reader.fieldnames) + ["cleaned_html"]

        writer = csv.DictWriter(
            outfile,
            fieldnames=fieldnames,
        )

        writer.writeheader()

        for row in reader:
            normalized_url = normalize_url(row["url"])

            # Skip duplicate URLs
            if normalized_url in seen_urls:
                continue

            seen_urls.add(normalized_url)

            html = row["raw_text"]

            html_body = extract_tag(html)
            body_text = extract_text(html_body)

            # Add spacing before each [Event block
            body_text = format_events(body_text)
            body_text = remove_odd_characters(body_text)

            row["url"] = normalized_url
            row["cleaned_html"] = body_text

            writer.writerow(row)

In [36]:
CLEANED_CRAWLED_DATA_PATH = "cleaned_crawled_data.csv"

In [37]:
clean_html_csv(
    input_csv=CRAWLED_DATA_PATH,
    output_csv=CLEANED_CRAWLED_DATA_PATH,
)

## 3.2. CSV (URL, HTML) -> Image CSV

In [38]:
import csv
import re
from urllib.parse import urljoin

from bs4 import BeautifulSoup


def extract_images_csv(
    input_csv: str,
    output_csv: str,
    content_tag="main",
):
    """
    Creates one row per image found in the HTML.

    Output columns:
        page_url: URL of the page where the image was found
        image_url: Absolute URL of the image
        alt: Alt text (if present)
    """

    with open(input_csv, "r", encoding="utf-8", newline="") as infile, \
         open(output_csv, "w", encoding="utf-8", newline="") as outfile:

        reader = csv.DictReader(infile)

        writer = csv.DictWriter(
            outfile,
            fieldnames=[
                "page_url",
                "image_url",
                "alt",
            ],
        )

        writer.writeheader()

        for row in reader:
            page_url = row["url"]
            html = row["raw_text"]

            soup = BeautifulSoup(html, "html.parser")

            main = soup.find(content_tag)

            if main is None:
                continue

            for image in main.find_all("img"):

                src = image.get("src")

                if not src:
                    continue

                absolute_url = urljoin(page_url, src)

                writer.writerow({
                    "page_url": page_url,
                    "image_url": absolute_url,
                    "alt": image.get("alt", ""),
                })

In [39]:
IMAGES_CSV = "website_images.csv"

In [40]:
extract_images_csv(CRAWLED_DATA_PATH, IMAGES_CSV)

## 4. Cleaned CSV -> Chunks

In [41]:
import csv

from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.core.schema import Document
from llama_index.embeddings.openai import OpenAIEmbedding


def create_chunked_csv(
    input_csv: str,
    output_csv: str,
    breakpoint_percentile_threshold=97
):
    embed_model = OpenAIEmbedding(
        model="text-embedding-3-large",
    )

    splitter = SemanticSplitterNodeParser(
        embed_model=embed_model,
        breakpoint_percentile_threshold=breakpoint_percentile_threshold,
    )

    with open(input_csv, "r", encoding="utf-8", newline="") as infile:
        reader = csv.DictReader(infile)

        with open(output_csv, "w", encoding="utf-8", newline="") as outfile:
            writer = csv.DictWriter(
                outfile,
                fieldnames=["url", "chunk_id", "chunk_text"],
            )

            writer.writeheader()

            for row in reader:
                url = row["url"]
                text = row["cleaned_html"]

                if not text.strip():
                    continue

                document = Document(text=text)

                nodes = splitter.get_nodes_from_documents([document])

                for chunk_id, node in enumerate(nodes):
                    writer.writerow(
                        {
                            "url": url,
                            "chunk_id": chunk_id,
                            "chunk_text": node.text,
                        }
                    )

In [42]:
CHUNKED_DATA_PATH = "chunked_data.csv"

In [43]:
create_chunked_csv(
    input_csv=CLEANED_CRAWLED_DATA_PATH,
    output_csv=CHUNKED_DATA_PATH,
)

## 5. Chunks -> Embeddings

In [44]:
EMBEDDING_CSV="embedded_chunked_data.csv"

In [ ]:
import json
import csv
import os

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

EMBEDDING_MODEL = "text-embedding-3-large"
client = OpenAI()
BATCH_SIZE = 100

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)


def get_embeddings(texts, embedding_model=EMBEDDING_MODEL):
    response = client.embeddings.create(
        model=embedding_model,
        input=texts,
    )

    return [item.embedding for item in response.data]


def write_batch(writer, rows, texts):
    embeddings = get_embeddings(texts)

    for row, embedding in zip(rows, embeddings):
        row["embedding"] = json.dumps(embedding)
        writer.writerow(row)

    print(f"Processed {len(rows)} rows")


def process_csv(input_csv=CHUNKED_DATA_PATH, output_csv=EMBEDDING_CSV, batch_size=BATCH_SIZE):
    with open(input_csv, "r", encoding="utf-8", newline="") as infile:
        reader = csv.DictReader(infile)

        fieldnames = list(reader.fieldnames or [])

        # Ensure embedding column exists exactly once
        fieldnames = [f for f in fieldnames if f != "embedding"]
        fieldnames.append("embedding")

        with open(output_csv, "w", encoding="utf-8") as outfile:
            writer = csv.DictWriter(outfile, fieldnames=fieldnames)
            writer.writeheader()

            batch_rows = []
            batch_texts = []

            for row in reader: 
                batch_rows.append(row)
                batch_texts.append(row.get("chunk_text", ""))

                if len(batch_rows) >= batch_size:
                    write_batch(
                        writer,
                        batch_rows,
                        batch_texts
                    )

                    batch_rows = []
                    batch_texts = []

            # Process remaining rows
            if batch_rows:
                write_batch(
                    writer,
                    batch_rows,
                    batch_texts
                )

    print(f"Finished. Output saved to {output_csv}")

In [46]:
process_csv()

Processed 100 rows
Processed 100 rows
Processed 100 rows
Processed 71 rows
Finished. Output saved to embedded_chunked_data.csv
